[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/17.%20The%20Container%20Reshuffling%20%28Remarshalling%29%20Problem/P17-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 17. The Container Reshuffling (Remarshalling) Problem

## Problem introduction

A container yard stores containers in stacks. A target container can be retrieved only when every container above it has been moved. This notebook studies a **multi-agent simulation** of that reshuffling process. Crane agents bid for movement tasks, stack agents evaluate local placement choices, and a coordinator supports system-level bookkeeping.

The method taught here is agent-based coordination with a simplified, highest-bid-wins auction. It is a heuristic simulation, not an exact optimization model: its utility scores and bids guide local decisions, but they do not establish a globally optimal reshuffling plan.

## What the notebook does

- **Inputs:** A fixed list of containers, the number of stacks, stack capacity, and the number of cranes.
- **Process:** Containers are assigned to stacks, target containers are retrieved when accessible, and blocking containers are moved to another stack when space is available. An auction selects an idle crane for each simulated movement.
- **Outputs:** The simulation reports retrievals, reshuffles, crane workloads, stack utilities, and activity over time.

## Multi-agent system components

1. **Crane agents** calculate bids and estimated movement times.
2. **Stack agents** evaluate local placement utility and track the state of their stack.
3. **A coordinator agent** maintains a system-level utility summary and supports message-based coordination.
4. **An auction** selects the crane with the highest bid for a reshuffle task, subject to a minimum bid threshold.

The code uses Belief-Desire-Intention (BDI) terminology to organize agent state. In this simulation, beliefs are stored observations, desires are named goals, and intentions are pending actions or plans; the notebook does not implement a complete BDI reasoning engine.

## Modeling assumptions and scope

- Each stack is ordered from bottom to top, and only its top container can be moved or retrieved.
- A reshuffle moves one blocking container to one other stack with available capacity.
- The destination rule prefers nearby stacks with more free space and fewer target containers.
- Crane bids are heuristic scores based on skill, stack-index distance, container priority, and workload. A higher bid is preferred; it is not a monetary payment or a proof of lower system-wide cost.
- The simulation is synchronous and simplified. It does not model physical travel paths, crane interference, safety constraints, container weight limits, handling times in the step loop, failures, or uncertainty in operations.
- Agent messages and auction timing fields illustrate coordination concepts. Movement auctions in the simulation are resolved directly by the system rather than through a distributed message broker.

## Decision logic

For a movement request \(r\), each idle crane \(k\) computes a bid \(b_k(r)\). The auction selects

$$
k^* \in \operatorname*{arg\,max}_{k \in K_{\mathrm{idle}}} b_k(r),
$$

provided that the winning bid meets the minimum threshold. If no crane is idle, or the winning bid is below the threshold, the movement is not carried out.

Here, \(K_{\mathrm{idle}}\) is the set of idle cranes and \(k^*\) is the selected crane. This rule is a local task-allocation heuristic. It does not minimize the total number of reshuffles or prove that a chosen sequence is globally optimal. The simulation proceeds in stages: initialize agents and stacks, place containers, repeatedly retrieve accessible targets or auction a move for a blocking container, then summarize the resulting activity.

In [ ]:
from typing import Any, Dict, List, Optional, Tuple

from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class AgentType(Enum):
    """Roles used to distinguish the agents in the simulation."""

    CRANE = "crane"
    STACK = "stack"
    COORDINATOR = "coordinator"
    OPTIMIZER = "optimizer"


@dataclass
class Message:
    """A message record for agent-to-agent communication."""

    sender_id: str
    receiver_id: str
    message_type: str
    content: Dict[str, Any]
    timestamp: datetime
    priority: int = 1


@dataclass
class Bid:
    """A crane's scored offer to perform a container movement."""

    agent_id: str
    container_id: int
    source_stack: int
    target_stack: int
    bid_value: float
    confidence: float
    estimated_time: float


@dataclass
class Container:
    """Container attributes and its current stack location."""

    id: int
    weight: float
    destination: str
    priority: int
    is_target: bool
    location: Optional[Tuple[int, int]] = None
    owner_agent: Optional[str] = None


@dataclass
class Stack:
    """A bottom-to-top container stack managed by a stack agent."""

    id: int
    max_height: int
    containers: List[Container] = field(default_factory=list)
    manager_agent: Optional[str] = None
    local_utility: float = 0.0

    def __post_init__(self) -> None:
        if self.max_height <= 0:
            raise ValueError("max_height must be a positive integer")
        self._update_utility()

    def add_container(self, container: Container) -> bool:
        """Place a container on top if the stack has capacity."""
        if self.is_full():
            return False

        container.location = (self.id, len(self.containers))
        self.containers.append(container)
        self._update_utility()
        return True

    def remove_top_container(self) -> Optional[Container]:
        """Remove and return the top container, if the stack is nonempty."""
        if not self.containers:
            return None

        container = self.containers.pop()
        container.location = None
        self._update_utility()
        return container

    def get_top_container(self) -> Optional[Container]:
        """Return the top container without changing the stack."""
        return self.containers[-1] if self.containers else None

    def is_full(self) -> bool:
        return len(self.containers) >= self.max_height

    def get_available_space(self) -> int:
        return self.max_height - len(self.containers)

    def _update_utility(self) -> None:
        """Score target accessibility and remaining stack capacity."""
        utility = 0.0
        for index, container in enumerate(self.containers):
            if container.is_target:
                blockers = len(self.containers) - index - 1
                utility -= blockers * 10

        utility += self.get_available_space() * 5
        utilization = len(self.containers) / self.max_height
        if 0.3 <= utilization <= 0.7:
            utility += 10

        self.local_utility = utility

In [ ]:
class Agent:
    """Base class with message, belief, desire, and intention state."""

    def __init__(self, agent_id: str, agent_type: AgentType):
        self.agent_id = agent_id
        self.agent_type = agent_type
        self.beliefs: Dict[str, Any] = {}
        self.desires: List[str] = []
        self.intentions: List[Any] = []
        self.message_queue = deque()
        self.neighbors: List[str] = []
        self.performance_history: List[Dict[str, Any]] = []

    def receive_message(self, message: Message) -> None:
        self.message_queue.append(message)

    def send_message(
        self,
        receiver_id: str,
        message_type: str,
        content: Dict[str, Any],
        priority: int = 1,
    ) -> Message:
        return Message(
            sender_id=self.agent_id,
            receiver_id=receiver_id,
            message_type=message_type,
            content=content,
            timestamp=datetime.now(),
            priority=priority,
        )

    def process_messages(self) -> None:
        while self.message_queue:
            self.handle_message(self.message_queue.popleft())

    def handle_message(self, message: Message) -> None:
        """Subclasses define how their role responds to messages."""
        del message

    def decide_action(self) -> Optional[Any]:
        return None

    def update_beliefs(self, observation: Dict[str, Any]) -> None:
        self.beliefs.update(observation)


class CraneAgent(Agent):
    """Crane agent that scores movement requests and estimates duration."""

    def __init__(self, agent_id: str, crane_id: int, location: Tuple[int, int]):
        super().__init__(agent_id, AgentType.CRANE)
        self.crane_id = crane_id
        self.location = location
        self.status = "idle"
        self.current_container: Optional[Container] = None
        self.skill_level = random.uniform(0.7, 1.0)
        self.workload = 0
        self.desires = ["minimize_idle_time", "maximize_throughput", "maintain_safety"]

    def handle_message(self, message: Message) -> None:
        if message.message_type == "movement_request" and self.status == "idle":
            request = message.content
            response = self.send_message(
                receiver_id=message.sender_id,
                message_type="movement_bid",
                content={
                    "bid_value": self._calculate_bid(request),
                    "crane_id": self.crane_id,
                    "estimated_time": self._estimate_movement_time(request),
                    "original_request": request,
                },
            )
            self.intentions.append(response)
        elif message.message_type == "status_inquiry":
            self.intentions.append(
                self.send_message(
                    receiver_id=message.sender_id,
                    message_type="status_response",
                    content={
                        "status": self.status,
                        "location": self.location,
                        "workload": self.workload,
                        "skill_level": self.skill_level,
                    },
                )
            )

    def _calculate_bid(self, movement_request: Dict[str, Any]) -> float:
        """Return a heuristic score; larger scores are preferred by the auction."""
        source_stack = movement_request["source_stack"]
        target_stack = movement_request["target_stack"]
        priority = movement_request.get("priority", 2)

        base_bid = self.skill_level * 100
        distance_penalty = abs(source_stack - target_stack) * 5
        priority_bonus = (4 - priority) * 20
        workload_penalty = self.workload * 2
        return max(0.0, base_bid - distance_penalty + priority_bonus - workload_penalty)

    def _estimate_movement_time(self, movement_request: Dict[str, Any]) -> float:
        """Estimate travel time in seconds using stack-index distance."""
        distance = abs(
            movement_request["source_stack"] - movement_request["target_stack"]
        )
        return distance * 30 / self.skill_level


class StackAgent(Agent):
    """Agent that evaluates placements and monitors one stack's utility."""

    def __init__(self, agent_id: str, stack: Stack):
        super().__init__(agent_id, AgentType.STACK)
        self.stack = stack
        self.stack.manager_agent = agent_id
        self.desires = ["minimize_blocking", "optimize_space", "facilitate_retrievals"]
        self.rearrangement_threshold = 0.3

    def handle_message(self, message: Message) -> None:
        if message.message_type == "utility_update":
            self.beliefs["global_utility"] = message.content.get("global_utility", 0.0)
        elif message.message_type == "container_placement_request":
            content = message.content
            utility = self._evaluate_placement_utility(
                content["container_id"], content.get("priority", 2)
            )
            self.intentions.append(
                self.send_message(
                    receiver_id=message.sender_id,
                    message_type="placement_response",
                    content={
                        "stack_id": self.stack.id,
                        "utility": utility,
                        "available_space": self.stack.get_available_space(),
                        "current_utility": self.stack.local_utility,
                    },
                )
            )

    def _evaluate_placement_utility(self, container_id: int, priority: int) -> float:
        """Score a proposed placement from this stack's local perspective."""
        del container_id
        if self.stack.is_full():
            return -1000.0

        utility = self.stack.get_available_space() * 10
        if self.stack.containers:
            top_priority = self.stack.containers[-1].priority
            utility += 20 if priority >= top_priority else -10
        if any(container.is_target for container in self.stack.containers):
            utility -= 30
        return utility

    def decide_action(self) -> Optional[Dict[str, Any]]:
        threshold = -self.rearrangement_threshold * 100
        if self.stack.local_utility < threshold:
            return {
                "action_type": "rearrange",
                "stack_id": self.stack.id,
                "current_utility": self.stack.local_utility,
                "reason": "Local utility is below the configured threshold",
            }
        return None


class CoordinatorAgent(Agent):
    """Coordinator that records system utility and handles coordination messages."""

    def __init__(self, agent_id: str):
        super().__init__(agent_id, AgentType.COORDINATOR)
        self.active_auctions: Dict[str, Dict[str, Any]] = {}
        self.auction_history: List[Dict[str, Any]] = []
        self.global_utility = 0.0
        self.auction_timeout = 30
        self.min_bid_threshold = 10.0
        self.desires = ["maximize_global_utility", "minimize_conflicts", "ensure_fairness"]

    def handle_message(self, message: Message) -> None:
        if message.message_type == "placement_response":
            stack_id = message.content["stack_id"]
            self.beliefs[f"stack_{stack_id}_utility"] = message.content["utility"]
        elif message.message_type == "movement_bid":
            auction_id = message.content.get("auction_id")
            if auction_id in self.active_auctions:
                self.active_auctions[auction_id]["bids"].append(message)

    def update_global_utility(self, stacks: List[Stack]) -> float:
        if not stacks:
            self.global_utility = 0.0
        else:
            self.global_utility = sum(stack.local_utility for stack in stacks) / len(stacks)
        return self.global_utility

In [ ]:
class MultiAgentContainerSystem:
    """Synchronous simulation of stack, crane, and coordinator agents."""

    def __init__(self, num_stacks: int, max_height: int, num_cranes: int):
        if num_stacks <= 0 or num_cranes <= 0:
            raise ValueError("num_stacks and num_cranes must be positive integers")

        self.num_stacks = num_stacks
        self.max_height = max_height
        self.num_cranes = num_cranes
        self.stacks = [Stack(index, max_height) for index in range(num_stacks)]
        self.agents: Dict[str, Agent] = {}
        self.containers: List[Container] = []
        self.target_containers: List[Container] = []
        self.current_time = 0
        self.total_reshuffles = 0
        self.total_retrievals = 0
        self.performance_history: List[Dict[str, Any]] = []
        self.auction_history: List[Dict[str, Any]] = []
        self._initialize_agents()

    def _initialize_agents(self) -> None:
        for stack in self.stacks:
            agent_id = f"stack_{stack.id}"
            self.agents[agent_id] = StackAgent(agent_id, stack)

        for index in range(self.num_cranes):
            agent_id = f"crane_{index}"
            self.agents[agent_id] = CraneAgent(
                agent_id, index, (index * 100, 50)
            )

        self.agents["coordinator_0"] = CoordinatorAgent("coordinator_0")
        self._establish_connections()

    def _establish_connections(self) -> None:
        coordinator = self.agents["coordinator_0"]
        for agent_id, agent in self.agents.items():
            if agent_id != coordinator.agent_id:
                agent.neighbors.append(coordinator.agent_id)
                coordinator.neighbors.append(agent_id)

        for stack_id in range(self.num_stacks):
            stack_agent = self.agents[f"stack_{stack_id}"]
            for neighbor_id in range(max(0, stack_id - 1), min(self.num_stacks, stack_id + 2)):
                if neighbor_id != stack_id:
                    stack_agent.neighbors.append(f"stack_{neighbor_id}")

    def add_containers(self, containers: List[Container]) -> None:
        if sum(stack.max_height for stack in self.stacks) < len(containers):
            raise ValueError("The total stack capacity is smaller than the container count")
        self.containers = list(containers)
        self.target_containers = [container for container in self.containers if container.is_target]
        self._initial_placement()

    def _initial_placement(self) -> None:
        """Place containers in input order using the highest local utility."""
        for container in self.containers:
            candidates = [
                (stack.id, self._evaluate_placement_for_stack(container, stack))
                for stack in self.stacks
                if not stack.is_full()
            ]
            if not candidates:
                raise RuntimeError(f"No available stack for container {container.id}")

            candidates.sort(key=lambda item: item[1], reverse=True)
            placed = self.stacks[candidates[0][0]].add_container(container)
            if not placed:
                raise RuntimeError(f"Placement failed for container {container.id}")

    def _evaluate_placement_for_stack(self, container: Container, stack: Stack) -> float:
        if stack.is_full():
            return -1000.0

        utility = stack.get_available_space() * 10
        if stack.containers:
            top_priority = stack.containers[-1].priority
            utility += 20 if container.priority >= top_priority else -10
        if container.is_target:
            utility += stack.get_available_space() * 15
        if any(existing.is_target for existing in stack.containers):
            utility -= 30
        return utility

    def _find_target_stack(self, target: Container) -> Optional[int]:
        for stack in self.stacks:
            if target in stack.containers:
                return stack.id
        return None

    def _find_best_destination(self, source_stack_id: int) -> Optional[int]:
        """Choose a feasible destination using distance, space, and target count."""
        candidates = []
        for stack in self.stacks:
            if stack.id == source_stack_id or stack.is_full():
                continue

            distance = abs(stack.id - source_stack_id)
            target_count = sum(container.is_target for container in stack.containers)
            score = -distance * 5 + stack.get_available_space() * 10 - target_count * 15
            candidates.append((score, stack.id))

        if not candidates:
            return None
        candidates.sort(key=lambda item: (item[0], -item[1]), reverse=True)
        return candidates[0][1]

    def _simulate_auction(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Award a movement to the idle crane with the highest eligible bid."""
        bids = []
        for index in range(self.num_cranes):
            crane = self.agents[f"crane_{index}"]
            if crane.status != "idle":
                continue

            bid_value = crane._calculate_bid(request)
            bids.append(
                {
                    "crane_id": crane.agent_id,
                    "bid_value": bid_value,
                    "estimated_time": crane._estimate_movement_time(request),
                }
            )

        coordinator = self.agents["coordinator_0"]
        if not bids:
            return {"success": False, "reason": "No idle crane is available", "total_bids": 0}

        bids.sort(key=lambda bid: bid["bid_value"], reverse=True)
        winning_bid = bids[0]
        auction_record = {
            "request": dict(request),
            "bids": bids,
            "winner": winning_bid["crane_id"],
            "resolved": winning_bid["bid_value"] >= coordinator.min_bid_threshold,
        }
        self.auction_history.append(auction_record)

        if not auction_record["resolved"]:
            return {
                "success": False,
                "reason": "The highest bid is below the minimum threshold",
                "total_bids": len(bids),
            }

        winning_crane = self.agents[winning_bid["crane_id"]]
        winning_crane.status = "busy"
        winning_crane.workload += 1
        winning_crane.status = "idle"

        return {
            "success": True,
            "winning_crane": winning_bid["crane_id"],
            "bid_value": winning_bid["bid_value"],
            "estimated_time": winning_bid["estimated_time"],
            "total_bids": len(bids),
        }

    def _coordinate_reshuffle(self, target_stack_id: int) -> Dict[str, Any]:
        """Auction one move to uncover the target in the specified stack."""
        target_stack = self.stacks[target_stack_id]
        top_container = target_stack.get_top_container()
        if top_container is None or top_container.is_target:
            return {"success": False, "reason": "No blocking container is available"}

        destination_id = self._find_best_destination(target_stack_id)
        if destination_id is None:
            return {"success": False, "reason": "No destination stack has free capacity"}

        request = {
            "container_id": top_container.id,
            "source_stack": target_stack_id,
            "target_stack": destination_id,
            "priority": top_container.priority,
            "confidence": 0.8,
            "reason": "target_access",
        }
        auction_result = self._simulate_auction(request)
        if not auction_result["success"]:
            return auction_result

        moved_container = target_stack.remove_top_container()
        if moved_container is None or not self.stacks[destination_id].add_container(moved_container):
            raise RuntimeError("A validated reshuffle could not be completed")

        return {
            "success": True,
            "reshuffles": 1,
            "auctions_resolved": 1,
            "winning_crane": auction_result["winning_crane"],
            "container_id": moved_container.id,
            "from_stack": target_stack_id,
            "to_stack": destination_id,
        }

    def simulate_step(self) -> Dict[str, Any]:
        """Retrieve accessible targets and attempt one move per blocked target."""
        step_start = time.time()
        step_results = {
            "reshuffles": 0,
            "retrievals": 0,
            "auctions_resolved": 0,
            "agent_actions": [],
            "global_utility": 0.0,
        }

        for target in self.target_containers[:]:
            target_stack_id = self._find_target_stack(target)
            if target_stack_id is None:
                continue

            target_stack = self.stacks[target_stack_id]
            if target_stack.get_top_container() is target:
                retrieved = target_stack.remove_top_container()
                if retrieved is not None:
                    self.target_containers.remove(target)
                    step_results["retrievals"] += 1
            else:
                result = self._coordinate_reshuffle(target_stack_id)
                if result["success"]:
                    step_results["reshuffles"] += result["reshuffles"]
                    step_results["auctions_resolved"] += result["auctions_resolved"]

        coordinator = self.agents["coordinator_0"]
        step_results["global_utility"] = coordinator.update_global_utility(self.stacks)

        for agent in self.agents.values():
            if agent.agent_type == AgentType.STACK:
                action = agent.decide_action()
                if action is not None:
                    step_results["agent_actions"].append(action)

        self.total_reshuffles += step_results["reshuffles"]
        self.total_retrievals += step_results["retrievals"]
        self.current_time += 1
        step_results["step_time"] = time.time() - step_start
        self.performance_history.append(step_results)
        return step_results

    def run_simulation(self, max_steps: int = 100) -> Dict[str, Any]:
        """Run the synchronous simulation until completion or the step limit."""
        if max_steps < 0:
            raise ValueError("max_steps must be nonnegative")

        start_time = time.time()
        for step in range(max_steps):
            step_result = self.simulate_step()
            if step % 20 == 0:
                print(
                    f"Step {step}: {step_result['retrievals']} retrievals, "
                    f"{step_result['reshuffles']} reshuffles, "
                    f"global utility {step_result['global_utility']:.1f}"
                )
            if not self.target_containers:
                print(f"All target containers were retrieved at step {step}.")
                break

        coordinator = self.agents["coordinator_0"]
        return {
            "simulation_time": time.time() - start_time,
            "total_steps": len(self.performance_history),
            "total_reshuffles": self.total_reshuffles,
            "total_retrievals": self.total_retrievals,
            "remaining_targets": len(self.target_containers),
            "final_global_utility": coordinator.global_utility,
            "performance_history": self.performance_history,
        }


print("Multi-agent simulation classes are ready.")

In [ ]:
containers = [
    Container(id=1, weight=20.5, destination="NYC", priority=2, is_target=False),
    Container(id=2, weight=18.2, destination="LAX", priority=1, is_target=True),
    Container(id=3, weight=22.1, destination="CHI", priority=3, is_target=False),
    Container(id=4, weight=19.8, destination="MIA", priority=2, is_target=False),
    Container(id=5, weight=21.3, destination="SEA", priority=1, is_target=True),
    Container(id=6, weight=17.9, destination="BOS", priority=2, is_target=False),
    Container(id=7, weight=23.4, destination="DEN", priority=3, is_target=False),
    Container(id=8, weight=20.1, destination="ATL", priority=2, is_target=False),
    Container(id=9, weight=19.5, destination="SFO", priority=1, is_target=True),
    Container(id=10, weight=21.8, destination="DAL", priority=2, is_target=False),
    Container(id=11, weight=20.3, destination="PHX", priority=1, is_target=True),
    Container(id=12, weight=22.7, destination="LAS", priority=3, is_target=False),
]

num_stacks = 6
max_height = 4
num_cranes = 3

mas = MultiAgentContainerSystem(num_stacks, max_height, num_cranes)
mas.add_containers(containers)

assert sum(len(stack.containers) for stack in mas.stacks) == len(containers)
assert all(len(stack.containers) <= stack.max_height for stack in mas.stacks)

print("Multi-agent system configuration")
print(f"Stacks: {num_stacks}; capacity per stack: {max_height}; cranes: {num_cranes}")
print(f"Containers: {len(containers)}; target containers: {len(mas.target_containers)}")
print(f"Agents: {len(mas.agents)}")

agent_types = defaultdict(int)
for agent in mas.agents.values():
    agent_types[agent.agent_type.value] += 1
print("Agent counts:")
for agent_type, count in sorted(agent_types.items()):
    print(f"  {agent_type}: {count}")

print("\nInitial yard state (levels are listed bottom to top)")
for stack in mas.stacks:
    print(
        f"Stack {stack.id}: {len(stack.containers)}/{max_height} containers, "
        f"utility {stack.local_utility:.1f}"
    )
    for level, container in enumerate(stack.containers):
        status = "target" if container.is_target else "regular"
        print(f"  Level {level}: container {container.id} ({status}, priority {container.priority})")

In [ ]:
simulation_result = mas.run_simulation(max_steps=80)

print("\nSimulation results")
print(f"Simulation time: {simulation_result['simulation_time']:.3f} seconds")
print(f"Steps completed: {simulation_result['total_steps']}")
print(f"Reshuffles: {simulation_result['total_reshuffles']}")
print(f"Target retrievals: {simulation_result['total_retrievals']}")
print(f"Targets remaining: {simulation_result['remaining_targets']}")
print(f"Final average stack utility: {simulation_result['final_global_utility']:.1f}")

if simulation_result["total_reshuffles"] > 0:
    descriptive_ratio = (
        simulation_result["total_retrievals"] / simulation_result["total_reshuffles"]
    )
    print(f"Descriptive retrievals-per-reshuffle ratio: {descriptive_ratio:.2f}")
else:
    print("No reshuffles were recorded, so the retrievals-per-reshuffle ratio is undefined.")

history = simulation_result["performance_history"]
if history:
    step_count = len(history)
    print("Average activity per simulated step")
    print(f"  Reshuffles: {sum(step['reshuffles'] for step in history) / step_count:.2f}")
    print(f"  Retrievals: {sum(step['retrievals'] for step in history) / step_count:.2f}")
    print(f"  Average stack utility: {sum(step['global_utility'] for step in history) / step_count:.2f}")

assert simulation_result["total_retrievals"] + simulation_result["remaining_targets"] == len(
    [container for container in containers if container.is_target]
)

In [ ]:
performance_history = simulation_result["performance_history"]
steps = list(range(len(performance_history)))

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

cumulative_reshuffles = np.cumsum([step["reshuffles"] for step in performance_history])
cumulative_retrievals = np.cumsum([step["retrievals"] for step in performance_history])
ax1.plot(steps, cumulative_reshuffles, linewidth=2, label="Cumulative reshuffles")
ax1.plot(steps, cumulative_retrievals, linewidth=2, label="Cumulative retrievals")
ax1.set_xlabel("Simulation step")
ax1.set_ylabel("Cumulative count")
ax1.set_title("Cumulative activity")
ax1.legend()
ax1.grid(True, alpha=0.3)

utilities = [step["global_utility"] for step in performance_history]
ax2.plot(steps, utilities, color="tab:red", linewidth=2)
ax2.set_xlabel("Simulation step")
ax2.set_ylabel("Average stack utility (score)")
ax2.set_title("Average stack utility over time")
ax2.grid(True, alpha=0.3)
if utilities:
    ax2.fill_between(steps, min(utilities), utilities, alpha=0.2, color="tab:red")

reshuffles_per_step = [step["reshuffles"] for step in performance_history]
retrievals_per_step = [step["retrievals"] for step in performance_history]
auctions_per_step = [step["auctions_resolved"] for step in performance_history]
ax3.bar(steps, reshuffles_per_step, alpha=0.7, label="Reshuffles")
ax3.bar(steps, retrievals_per_step, alpha=0.7, label="Retrievals")
ax3.plot(steps, auctions_per_step, "ro-", linewidth=1.5, markersize=4, label="Resolved auctions")
ax3.set_xlabel("Simulation step")
ax3.set_ylabel("Count per step")
ax3.set_title("Activity by step")
ax3.legend()
ax3.grid(True, alpha=0.3)

crane_ids = [f"Crane {index}" for index in range(num_cranes)]
crane_workloads = [mas.agents[f"crane_{index}"].workload for index in range(num_cranes)]
bars = ax4.bar(crane_ids, crane_workloads, color="tab:orange", alpha=0.75)
ax4.set_xlabel("Crane agent")
ax4.set_ylabel("Assigned movement tasks")
ax4.set_title("Crane workload")
ax4.grid(True, axis="y", alpha=0.3)
for bar, workload in zip(bars, crane_workloads):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(workload),
        ha="center",
        va="bottom",
    )

fig.tight_layout()
plt.show()

print("\nFinal stack utility scores")
final_stack_utilities = [(stack.id, stack.local_utility) for stack in mas.stacks]
for stack_id, utility in final_stack_utilities:
    print(f"  Stack {stack_id}: {utility:.1f}")

utility_values = [utility for _, utility in final_stack_utilities]
if utility_values:
    average_utility = float(np.mean(utility_values))
    utility_variance = float(np.var(utility_values))
    print(f"Mean stack utility: {average_utility:.2f}")
    print(f"Stack-utility variance: {utility_variance:.2f}")
    print(
        "Variance describes dispersion in the defined utility scores; by itself, "
        "it does not establish equilibrium or system optimality."
    )

In [ ]:
print("Coordination diagnostics")

resolved_auctions = [auction for auction in mas.auction_history if auction["resolved"]]
rejected_auctions = [auction for auction in mas.auction_history if not auction["resolved"]]
print(f"Resolved auctions: {len(resolved_auctions)}")
print(f"Auctions below the bid threshold: {len(rejected_auctions)}")

if resolved_auctions:
    average_bids = sum(len(auction["bids"]) for auction in resolved_auctions) / len(
        resolved_auctions
    )
    print(f"Average bids per resolved auction: {average_bids:.2f}")
else:
    print("No auction was resolved, so an average bid count is not available.")

if crane_workloads:
    largest_workload = max(crane_workloads)
    smallest_workload = min(crane_workloads)
    if largest_workload > 0:
        workload_balance = 1 - (largest_workload - smallest_workload) / largest_workload
    else:
        workload_balance = 1.0
    print(f"Crane workload range: {smallest_workload} to {largest_workload} tasks")
    print(f"Workload balance index: {workload_balance:.3f}")

print(
    "These diagnostics describe this run of the heuristic simulation. They do not "
    "measure global optimality, prove convergence, or establish a Nash equilibrium."
)

## Summary and key insights

### What the simulation demonstrates

- **Role-based agents:** Crane, stack, and coordinator agents store different state and perform different tasks.
- **Auction-based task allocation:** Idle cranes score a movement request, and the highest eligible bid wins.
- **Local utility:** Stack scores combine target accessibility, free capacity, and a simple utilization preference. They are modeling scores, not directly measured costs or benefits.
- **Reshuffling and retrieval:** Only top containers can be moved or retrieved. A target blocked by a non-target container may require a reshuffle before retrieval.
- **Run-specific diagnostics:** Counts, utility trajectories, and workload distributions help describe the simulated run, but do not establish global optimality.

### Interpreting the results

The number of reshuffles depends on the initial placement, the destination-stack rule, the heuristic bids, and the sequence in which targets are considered. The workload chart shows how movement assignments were distributed among cranes in this run. A low stack-utility variance means the defined scores are similar; it is not evidence that the system has reached a Nash equilibrium or an optimal arrangement.

The simulation is intentionally narrower than a real terminal. It assumes a single top-accessible stack operation at a time and omits physical travel constraints, interference between cranes, safety rules, and stochastic operational failures. Its auction and utility rules should therefore be interpreted as a transparent teaching model of agent coordination, not as a validated operational policy.

### Further study

Natural extensions include adding operational constraints to movement feasibility, examining how the existing heuristic rules behave under different yard layouts, and recording additional measures such as movement time or blocked-target counts. Such extensions would require explicit assumptions and evaluation criteria before their results could support operational conclusions.